# Data Cleaning & Preparation: E-commerce Orders

**DecodeLabs Data Analytics Internship, Project 1**  |  Prepared by Muhammad Waqar Tahir

**Goal:** clean a raw dataset by handling missing values, duplicates and incorrect data, then prove the result is reliable.

**Approach:** load everything as text so nothing is silently converted, audit first, clean in a fixed order, save, then re-read the saved file and verify it against the guide's gate (zero duplicate IDs, zero incorrectly formatted dates).

| Step | What it does |
|---|---|
| 1 | Load and inspect the raw data |
| 2 | Audit: measure what is wrong before touching anything |
| 3 | Clean: whitespace, missing values, case, dates, numbers, duplicates |
| 4 | Save the cleaned file |
| 5 | Verification gate on the saved file |
| 6 | Change log and observations |

In [1]:
import sys
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

# Works whether Jupyter starts in the repo root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = ROOT / "data" / "raw" / "orders_raw.csv"
CLEAN_PATH = ROOT / "data" / "cleaned" / "orders_cleaned.csv"

change_log = []   # every step appends: (step, what changed, records affected)
print("Raw file:", RAW_PATH.relative_to(ROOT))

Raw file: data/raw/orders_raw.csv


## 1. Load and inspect

Every column is read as **text** (`dtype=str`) and pandas is told not to turn blanks into `NaN`. That way blanks stay visible as blanks and no value is reinterpreted before I have looked at it.

In [2]:
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head(8)

1,200 rows x 14 columns


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.1
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.7
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.4
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
5,ORD200005,2023-10-23,C37249,Phone,2,245.86,934 Main St,Credit Card,Shipped,TRK72976927,4,SAVE10,Instagram,491.72
6,ORD200006,2025-06-17,C83492,Laptop,1,664.42,986 Main St,Gift Card,Returned,TRK96417362,6,SAVE10,Facebook,664.42
7,ORD200007,2023-05-12,C41460,Monitor,5,149.55,706 Main St,Cash,Shipped,TRK78809193,9,FREESHIP,Facebook,747.75


In [3]:
raw.describe().T[["count", "unique", "top", "freq"]]

,count,unique,top,freq
OrderID,1200,1200,ORD200000,1
Date,1200,671,2023-08-20,8
CustomerID,1200,1189,C98474,2
Product,1200,7,Printer,181
Quantity,1200,5,1,255
UnitPrice,1200,1193,127.18,2
ShippingAddress,1200,655,533 Main St,8
PaymentMethod,1200,5,Online,258
OrderStatus,1200,5,Cancelled,250
TrackingNumber,1200,1200,TRK37947903,1


## 2. Audit (read-only)

Measure every problem type from the guide **before** changing anything, so the change log can be checked against real numbers.

In [4]:
NULL_TOKENS = {"", "nan", "null", "none", "n/a", "na", "-", "?"}

def is_null(s):
    return s.str.strip().str.lower().isin(NULL_TOKENS)

missing = pd.DataFrame({
    "missing": {c: int(is_null(raw[c]).sum()) for c in raw.columns},
})
missing["% of rows"] = (missing["missing"] / len(raw) * 100).round(2)
missing[missing["missing"] > 0]

,missing,% of rows
CouponCode,309,25.75


Only **`CouponCode`** has gaps: 309 blanks, about a quarter of all orders. Every other column is complete.

In [5]:
ISO_DATE = re.compile(r"^\d{4}-\d{2}-\d{2}$")
TWO_DP = re.compile(r"^\d+\.\d{2}$")
ID_PATTERNS = {"OrderID": r"^ORD\d{6}$", "CustomerID": r"^C\d{5}$", "TrackingNumber": r"^TRK\d{8}$"}
TEXT_COLS = ["Product", "PaymentMethod", "OrderStatus", "ReferralSource", "ShippingAddress"]
PRICE_COLS = ["UnitPrice", "TotalPrice"]
INT_COLS = ["Quantity", "ItemsInCart"]

def to_number(s):
    return pd.to_numeric(s.str.replace(r"[^\d.\-]", "", regex=True), errors="coerce")

qty, unit, total = (to_number(raw[c]) for c in ("Quantity", "UnitPrice", "TotalPrice"))

audit = pd.Series({
    "Duplicate rows (all columns)":          raw.duplicated().sum(),
    "Duplicate OrderIDs":                    raw.duplicated("OrderID").sum(),
    "Duplicate TrackingNumbers":             raw.duplicated("TrackingNumber").sum(),
    "Dates not in YYYY-MM-DD":               (~raw["Date"].str.match(ISO_DATE)).sum(),
    "Cells with stray whitespace":           sum((raw[c] != raw[c].str.strip()).sum() for c in raw.columns),
    "Text cells not in Title Case":          sum((raw[c] != raw[c].str.title()).sum() for c in TEXT_COLS),
    "IDs not matching expected pattern":     sum((~raw[c].str.match(p)).sum() for c, p in ID_PATTERNS.items()),
    "Non-numeric numbers":                   sum(to_number(raw[c]).isna().sum() for c in INT_COLS + PRICE_COLS),
    "Price cells not at exactly 2 decimals": sum((~raw[c].str.match(TWO_DP)).sum() for c in PRICE_COLS),
    "TotalPrice != Quantity x UnitPrice":    ((qty * unit).round(2) - total).abs().gt(0.011).sum(),
}).astype(int).to_frame("records")
audit

,records
Duplicate rows (all columns),0
Duplicate OrderIDs,0
Duplicate TrackingNumbers,0
Dates not in YYYY-MM-DD,0
Cells with stray whitespace,0
Text cells not in Title Case,0
IDs not matching expected pattern,0
Non-numeric numbers,0
Price cells not at exactly 2 decimals,374
TotalPrice != Quantity x UnitPrice,0


**Audit result.** The raw file is in good shape. The real issues are the 309 blank coupons and 374 price cells (spread across 260 rows) that are not written to two decimals, for example `224` instead of `224.00`. Everything else already passes, and the steps below still run so the pipeline is repeatable on messier data.

## 3. Clean

Work on a copy so `raw` stays untouched for comparison. Each step records how many records it changed.

In [6]:
df = raw.copy()

def log(step, what, records):
    change_log.append({"step": step, "what changed": what, "records affected": int(records)})
    print(f"{step:<28} records affected: {int(records):,}")

### Step 1: Whitespace
Trim leading/trailing spaces and collapse repeated spaces, so `"Laptop"` and `"Laptop "` cannot become two categories.

In [7]:
changed = pd.Series(False, index=df.index)
for c in df.columns:
    new = df[c].str.strip().str.replace(r"\s+", " ", regex=True)
    changed |= new != df[c]
    df[c] = new
log("1. Whitespace", "Trim / collapse spaces", changed.sum())

1. Whitespace                records affected: 0


### Step 2: Missing values

**`CouponCode`.** A blank here means *the customer used no coupon*, which is information, not lost data. Options considered:

| Option | Verdict |
|---|---|
| Delete the 309 rows | Rejected. Throws away 25.8% of orders (the guide warns listwise deletion reduces statistical power) |
| Fill with the mode (most common coupon) | Rejected. Invents discounts that never happened |
| Label as `No Coupon` | **Chosen.** Keeps every row and keeps the meaning |

In [8]:
NO_COUPON = "No Coupon"
coupon_blank = is_null(df["CouponCode"])
df.loc[coupon_blank, "CouponCode"] = NO_COUPON
log("2a. CouponCode blanks", "Blank -> 'No Coupon'", coupon_blank.sum())

df["CouponCode"].value_counts()

2a. CouponCode blanks        records affected: 309


CouponCode
FREESHIP     313
No Coupon    309
WINTER15     292
SAVE10       286
Name: count, dtype: int64

**All other columns.** None are missing, but the strategy from the guide is applied anyway so the pipeline is safe on new data: **median** for numbers, **mode** for categories, and `Quantity x UnitPrice` for a missing `TotalPrice` (an exact calculation beats a guess).

In [9]:
other_missing = 0
for c in TEXT_COLS + INT_COLS + PRICE_COLS:
    mask = is_null(df[c])
    if not mask.any():
        continue
    other_missing += int(mask.sum())
    if c == "TotalPrice":
        fill = (to_number(df["Quantity"]) * to_number(df["UnitPrice"])).round(2)[mask].astype(str)
    elif c in INT_COLS + PRICE_COLS:
        fill = str(to_number(df.loc[~mask, c]).median())
    else:
        fill = df.loc[~mask, c].mode().iloc[0]
    df.loc[mask, c] = fill
log("2b. Other missing values", "Median / mode / calculated", other_missing)

2b. Other missing values     records affected: 0


### Step 3: Consistent case
Title Case for descriptive text, UPPER CASE for identifiers and coupon codes. Coupon codes stay upper case (`SAVE10`) because they are codes, not words.

In [10]:
changed = pd.Series(False, index=df.index)
for c in TEXT_COLS:
    new = df[c].str.title()
    changed |= new != df[c]
    df[c] = new

keep = df["CouponCode"] != NO_COUPON
new = df["CouponCode"].where(~keep, df["CouponCode"].str.upper())
changed |= new != df["CouponCode"]
df["CouponCode"] = new

for c in ID_PATTERNS:
    new = df[c].str.upper()
    changed |= new != df[c]
    df[c] = new
log("3. Case", "Title Case text, UPPER ids/codes", changed.sum())

3. Case                      records affected: 0


### Step 4: Dates to ISO 8601
`YYYY-MM-DD` sorts correctly as text and is understood by Excel, SQL and pandas. Anything that fails to parse would show up as `NaT` and be caught by the verification gate.

In [11]:
parsed = pd.to_datetime(df["Date"], format="%Y-%m-%d", errors="coerce")
unparsed = df["Date"].where(parsed.isna() & ~is_null(df["Date"]))
if unparsed.notna().any():   # tolerant fallback; month-first for ambiguous strings like 05/02/2024
    parsed.loc[unparsed.notna()] = pd.to_datetime(unparsed.dropna(), format="mixed", errors="coerce")

iso = parsed.dt.strftime("%Y-%m-%d")
changed = iso.fillna("") != df["Date"]
df["Date"] = iso.fillna(df["Date"])
log("4. Dates", "Converted to YYYY-MM-DD", changed.sum())
print("Range:", df["Date"].min(), "to", df["Date"].max())

4. Dates                     records affected: 0
Range: 2023-01-01 to 2025-06-30


### Step 5: Numeric types and precision
Quantities become whole numbers and prices are fixed to exactly two decimals. This changes how a value is *written* (`224` to `224.00`), never the value itself. That is proved in the next cell.

In [12]:
before_values = {c: to_number(df[c]).copy() for c in INT_COLS + PRICE_COLS}

changed = pd.Series(False, index=df.index)
cells = {}
for c in INT_COLS:
    new = to_number(df[c]).round().astype("Int64").astype(str)
    changed |= new != df[c]
    df[c] = new
for c in PRICE_COLS:
    new = to_number(df[c]).map(lambda v: f"{v:.2f}" if pd.notna(v) else "")
    cells[c] = int((new != df[c]).sum())
    changed |= new != df[c]
    df[c] = new
log("5. Numeric formats", "2 decimal prices, integer counts", changed.sum())
print("Cells reformatted:", cells)

# Proof that no numeric value moved
moved = {c: int((to_number(df[c]) - before_values[c]).abs().gt(1e-9).sum()) for c in before_values}
print("Values numerically changed:", moved)
assert sum(moved.values()) == 0

5. Numeric formats           records affected: 260
Cells reformatted: {'UnitPrice': 114, 'TotalPrice': 260}


Values numerically changed: {'Quantity': 0, 'ItemsInCart': 0, 'UnitPrice': 0, 'TotalPrice': 0}


### Step 6: Duplicates
"One truth, one record." Remove exact duplicate rows first, then any repeated `OrderID` (first occurrence kept).

In [13]:
n0 = len(df)
df = df.drop_duplicates(keep="first")
full = n0 - len(df)
n1 = len(df)
df = df.drop_duplicates(subset="OrderID", keep="first")
by_id = n1 - len(df)
df = df.reset_index(drop=True)
log("6. Duplicates", f"Full-row: {full}, extra OrderID rows: {by_id}", full + by_id)
print(f"Rows: {len(raw):,} -> {len(df):,}")

6. Duplicates                records affected: 0
Rows: 1,200 -> 1,200


## 4. Save

UTF-8 with LF line endings (the raw file used Windows CRLF), no index column.

In [14]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False, encoding="utf-8", lineterminator="\n")
print("Saved:", CLEAN_PATH.relative_to(ROOT), f"({len(df):,} rows)")

Saved: data/cleaned/orders_cleaned.csv (1,200 rows)


## 5. Verification gate

The guide's rule: *"Before you finish, you must prove there are zero duplicate IDs and zero incorrectly formatted dates."*

To make the proof honest, the check reads the **saved file from disk** rather than the DataFrame still in memory.

In [15]:
check = pd.read_csv(CLEAN_PATH, dtype=str, keep_default_na=False)
dt = pd.to_datetime(check["Date"], format="%Y-%m-%d", errors="coerce")
q, u, t = (to_number(check[c]) for c in ("Quantity", "UnitPrice", "TotalPrice"))

results = pd.DataFrame([
    ("GATE  Duplicate OrderIDs",                  check.duplicated("OrderID").sum()),
    ("GATE  Incorrectly formatted dates",         (~check["Date"].str.match(ISO_DATE)).sum() + dt.isna().sum()),
    ("Duplicate rows (all columns)",              check.duplicated().sum()),
    ("Duplicate TrackingNumbers",                 check.duplicated("TrackingNumber").sum()),
    ("Missing / null-like values",                sum(is_null(check[c]).sum() for c in check.columns)),
    ("IDs not matching expected pattern",         sum((~check[c].str.match(p)).sum() for c, p in ID_PATTERNS.items())),
    ("Prices not at 2 decimals",                  sum((~check[c].str.match(TWO_DP)).sum() for c in PRICE_COLS)),
    ("Non-numeric numbers",                       sum(to_number(check[c]).isna().sum() for c in INT_COLS + PRICE_COLS)),
    ("TotalPrice != Quantity x UnitPrice",        ((q * u).round(2) - t).abs().gt(0.011).sum()),
    ("Row count differs from raw",                abs(len(check) - len(raw))),
], columns=["check", "failures"])
results["failures"] = results["failures"].astype(int)
results["result"] = results["failures"].map(lambda n: "PASS" if n == 0 else "FAIL")
results

,check,failures,result
0,GATE Duplicate OrderIDs,0,PASS
1,GATE Incorrectly formatted dates,0,PASS
2,Duplicate rows (all columns),0,PASS
3,Duplicate TrackingNumbers,0,PASS
4,Missing / null-like values,0,PASS
5,IDs not matching expected pattern,0,PASS
6,Prices not at 2 decimals,0,PASS
7,Non-numeric numbers,0,PASS
8,TotalPrice != Quantity x UnitPrice,0,PASS
9,Row count differs from raw,0,PASS


In [16]:
assert (results["failures"] == 0).all(), "Verification failed, see table above"
print("ALL CHECKS PASSED: 0 duplicate IDs, 0 incorrectly formatted dates")

ALL CHECKS PASSED: 0 duplicate IDs, 0 incorrectly formatted dates


## 6. Change log

In [17]:
log_df = pd.DataFrame(change_log)
log_df

,step,what changed,records affected
0,1. Whitespace,Trim / collapse spaces,0
1,2a. CouponCode blanks,Blank -> 'No Coupon',309
2,2b. Other missing values,Median / mode / calculated,0
3,3. Case,"Title Case text, UPPER ids/codes",0
4,4. Dates,Converted to YYYY-MM-DD,0
5,5. Numeric formats,"2 decimal prices, integer counts",260
6,6. Duplicates,"Full-row: 0, extra OrderID rows: 0",0


Steps with **0 records** were tested and found clean, so nothing needed fixing. The three real changes are the 309 coupon labels, the price reformatting, and (at file level) the line-ending normalisation. The full formal log, including audit IDs, is in `reports/Data_Cleaning_Change_Log.pdf`.

### Before and after

In [18]:
comparison = pd.DataFrame({
    "before": {"rows": len(raw), "missing values": int(missing["missing"].sum()),
               "price cells not at 2 dp": int(audit.loc["Price cells not at exactly 2 decimals", "records"]),
               "duplicate OrderIDs": int(raw.duplicated("OrderID").sum())},
    "after":  {"rows": len(check), "missing values": int(results.loc[4, "failures"]),
               "price cells not at 2 dp": int(results.loc[6, "failures"]),
               "duplicate OrderIDs": int(results.loc[0, "failures"])},
})
comparison

,before,after
rows,1200,1200
missing values,309,0
price cells not at 2 dp,374,0
duplicate OrderIDs,0,0


## 7. Observations (flagged, not altered)

These are properties of the source data that anyone analysing it should know about. They were **not** changed, because altering them would mean inventing data.

In [19]:
tp, up = to_number(check["TotalPrice"]), to_number(check["UnitPrice"])
voided = check["OrderStatus"].isin(["Cancelled", "Returned"])
tracked = check["OrderStatus"].isin(["Cancelled", "Pending"]) & check["TrackingNumber"].ne("")
per_customer = check["CustomerID"].value_counts()
lap = up[check["Product"] == "Laptop"]

print(f"1. TotalPrice equals Quantity x UnitPrice on all {len(check):,} rows, so coupon discounts and shipping")
print("   are not reflected in it.")
print(f"2. {voided.sum()} orders ({voided.mean():.1%}) are Cancelled or Returned but still carry a TotalPrice.")
print("   Filter on OrderStatus before calculating revenue.")
print(f"3. {tracked.sum()} Cancelled or Pending orders have a TrackingNumber (unusual for orders that never shipped).")
print(f"4. UnitPrice varies widely inside one product (Laptop: {lap.min():.2f} to {lap.max():.2f}).")
print(f"5. {(per_customer > 1).sum()} CustomerIDs appear on more than one order: repeat customers, not duplicates.")

1. TotalPrice equals Quantity x UnitPrice on all 1,200 rows, so coupon discounts and shipping
   are not reflected in it.
2. 497 orders (41.4%) are Cancelled or Returned but still carry a TotalPrice.
   Filter on OrderStatus before calculating revenue.
3. 487 Cancelled or Pending orders have a TrackingNumber (unusual for orders that never shipped).
4. UnitPrice varies widely inside one product (Laptop: 18.20 to 699.93).
5. 11 CustomerIDs appear on more than one order: repeat customers, not duplicates.


### Sanity check against the command-line script

The same logic also lives in `src/clean_data.py`. This cell confirms the notebook and the script produce exactly the same cleaned table.

In [20]:
sys.path.insert(0, str(ROOT / "src"))
import clean_data as cd

script_df, _ = cd.clean(cd.load_raw(RAW_PATH))
assert script_df.equals(check), "Notebook and script results differ"
print("Notebook output is identical to src/clean_data.py output.")

Notebook output is identical to src/clean_data.py output.


## Conclusion

- **1,200 rows in, 1,200 rows out.** No record was deleted.
- The one substantive gap (309 blank coupon codes) was labelled instead of deleted or guessed.
- Price formatting was standardised without changing any value.
- The saved file passes the verification gate: **0 duplicate IDs, 0 incorrectly formatted dates**, and 8 further checks.
- Judgement calls and data oddities are documented rather than silently "fixed".